<a href="https://colab.research.google.com/github/KhaledTachyonAlgernon/GrandNexus/blob/main/GRANDNEXUS_PHASE_2_INTELLIGENT_SYSTEM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

GRANDNEXUS PHASE 2 - INTELLIGENT SYSTEM

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
GRANDNEXUS - SYSTÈME INTELLIGENT COMPLET
Phase 1 (Corrigée) + Phase 2 (Intelligence)
"""

import asyncio
import logging
import time
import json
import sqlite3
import hashlib
import pickle
import re
from datetime import datetime, timedelta
from pathlib import Path
from typing import Dict, Any, List, Optional, Set, Tuple, Union
from dataclasses import dataclass, field, asdict
from collections import defaultdict, deque
from enum import Enum
import numpy as np
import uuid

# Configuration logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger("GrandNexus")

# ============================================================================
# DATACLASSES POUR LE RAISONNEMENT
# ============================================================================

class RelationType(str, Enum):
    """Types de relations logiques"""
    IS_A = "is_a"                    # "Chat is_a Animal"
    PART_OF = "part_of"              # "Roue part_of Voiture"
    HAS_PROPERTY = "has_property"    # "Socrate has_property mortel"
    CAUSES = "causes"                # "Pluie causes Humidité"
    IMPLIES = "implies"              # "A implies B"
    CONTRADICTS = "contradicts"      # "A contradicts B"
    SIMILAR_TO = "similar_to"        # "Chien similar_to Loup"
    CO_OCCURS = "co_occurs"          # Co-occurrence statistique
    PRECEDES = "precedes"            # Ordre temporel
    PROVIDES = "provides"            # Source de données

@dataclass
class LogicalFact:
    """Fait logique avec prédicat"""
    subject: str
    predicate: str  # RelationType
    object: str
    confidence: float = 1.0
    source: str = "system"
    timestamp: float = field(default_factory=time.time)

    def __str__(self):
        return f"{self.subject} --[{self.predicate}]--> {self.object} ({self.confidence:.2f})"

@dataclass
class InferenceRule:
    """Règle d'inférence"""
    name: str
    premises: List[str]  # Patterns de prémisses
    conclusion: str      # Pattern de conclusion
    confidence_decay: float = 0.9  # Perte de confiance lors de l'inférence

    def __str__(self):
        premises_str = " ∧ ".join(self.premises)
        return f"{self.name}: {premises_str} → {self.conclusion}"

@dataclass
class Query:
    """Requête utilisateur"""
    text: str
    query_type: str  # "question", "command", "statement"
    entities: List[str] = field(default_factory=list)
    intent: Optional[str] = None
    timestamp: float = field(default_factory=time.time)

@dataclass
class Answer:
    """Réponse du système"""
    text: str
    confidence: float
    reasoning_path: List[str] = field(default_factory=list)
    sources: List[str] = field(default_factory=list)
    metadata: Dict[str, Any] = field(default_factory=dict)

# ============================================================================
# REASONING ENGINE - MOTEUR DE RAISONNEMENT SYMBOLIQUE
# ============================================================================

class SymbolicReasoningEngine:
    """
    Moteur de raisonnement symbolique avec inférence logique

    Capacités:
    - Déduction (modus ponens, syllogisme)
    - Induction (généralisation)
    - Abduction (meilleure explication)
    - Raisonnement par analogie
    """

    def __init__(self, graph, bus):
        self.graph = graph  # Graphe de connaissances
        self.bus = bus

        # Base de faits logiques
        self.facts: List[LogicalFact] = []
        self.facts_index: Dict[str, List[LogicalFact]] = defaultdict(list)

        # Règles d'inférence
        self.rules: List[InferenceRule] = []
        self._init_default_rules()

        # Cache d'inférences
        self.inference_cache: Dict[str, List[LogicalFact]] = {}

        # Statistiques
        self.stats = {
            'total_inferences': 0,
            'cache_hits': 0,
            'deductions': 0,
            'inductions': 0,
            'abductions': 0
        }

        logger.info("✅ Symbolic Reasoning Engine initialized")

    def _init_default_rules(self):
        """Initialise les règles d'inférence de base"""

        # Transitivité pour is_a
        self.rules.append(InferenceRule(
            name="transitive_is_a",
            premises=[
                "?X is_a ?Y",
                "?Y is_a ?Z"
            ],
            conclusion="?X is_a ?Z",
            confidence_decay=0.95
        ))

        # Héritage de propriétés
        self.rules.append(InferenceRule(
            name="property_inheritance",
            premises=[
                "?X is_a ?Y",
                "?Y has_property ?P"
            ],
            conclusion="?X has_property ?P",
            confidence_decay=0.9
        ))

        # Transitivité pour part_of
        self.rules.append(InferenceRule(
            name="transitive_part_of",
            premises=[
                "?X part_of ?Y",
                "?Y part_of ?Z"
            ],
            conclusion="?X part_of ?Z",
            confidence_decay=0.95
        ))

        # Causalité transitive
        self.rules.append(InferenceRule(
            name="transitive_causes",
            premises=[
                "?X causes ?Y",
                "?Y causes ?Z"
            ],
            conclusion="?X causes ?Z",
            confidence_decay=0.85
        ))

        # Implication
        self.rules.append(InferenceRule(
            name="modus_ponens",
            premises=[
                "?X implies ?Y",
                "?X"
            ],
            conclusion="?Y",
            confidence_decay=0.95
        ))

        logger.info(f"📋 Loaded {len(self.rules)} inference rules")

    async def add_fact(self, fact: LogicalFact):
        """Ajoute un fait à la base de connaissances"""
        self.facts.append(fact)

        # Indexation pour recherche rapide
        self.facts_index[fact.subject].append(fact)
        self.facts_index[fact.object].append(fact)

        # Invalider le cache
        self.inference_cache.clear()

        logger.debug(f"➕ Fact added: {fact}")

    async def query_facts(self, subject: str = None, predicate: str = None,
                         object: str = None) -> List[LogicalFact]:
        """Recherche de faits par pattern matching"""
        results = []

        for fact in self.facts:
            if subject and fact.subject != subject:
                continue
            if predicate and fact.predicate != predicate:
                continue
            if object and fact.object != object:
                continue

            results.append(fact)

        return results

    async def deduce(self, max_depth: int = 3) -> List[LogicalFact]:
        """
        Déduction: applique les règles d'inférence pour découvrir de nouveaux faits

        Returns:
            Liste des nouveaux faits inférés
        """
        new_facts = []
        current_facts = self.facts.copy()

        for depth in range(max_depth):
            round_new_facts = []

            for rule in self.rules:
                # Applique chaque règle
                inferred = await self._apply_rule(rule, current_facts)
                round_new_facts.extend(inferred)

            if not round_new_facts:
                break  # Plus rien à inférer

            # Ajoute les nouveaux faits
            for fact in round_new_facts:
                if not self._fact_exists(fact):
                    await self.add_fact(fact)
                    new_facts.append(fact)

            current_facts = self.facts.copy()

            logger.info(f"🔍 Inference depth {depth+1}: {len(round_new_facts)} new facts")

        self.stats['deductions'] += len(new_facts)
        self.stats['total_inferences'] += len(new_facts)

        return new_facts

    async def _apply_rule(self, rule: InferenceRule, facts: List[LogicalFact]) -> List[LogicalFact]:
        """Applique une règle d'inférence aux faits"""
        inferred_facts = []

        if len(rule.premises) == 2:
            # Règle binaire (la plus courante)
            premise1_pattern = rule.premises[0]
            premise2_pattern = rule.premises[1]

            # Parse les patterns
            p1_parts = premise1_pattern.split()
            p2_parts = premise2_pattern.split()

            if len(p1_parts) == 3 and len(p2_parts) == 3:
                # Pattern: ?X relation ?Y
                for fact1 in facts:
                    if fact1.predicate != p1_parts[1]:
                        continue

                    for fact2 in facts:
                        if fact2.predicate != p2_parts[1]:
                            continue

                        # Unification des variables
                        bindings = {}

                        # Première prémisse
                        if p1_parts[0].startswith('?'):
                            bindings[p1_parts[0]] = fact1.subject
                        elif p1_parts[0] != fact1.subject:
                            continue

                        if p1_parts[2].startswith('?'):
                            bindings[p1_parts[2]] = fact1.object
                        elif p1_parts[2] != fact1.object:
                            continue

                        # Deuxième prémisse
                        if p2_parts[0].startswith('?'):
                            expected = bindings.get(p2_parts[0])
                            if expected and expected != fact2.subject:
                                continue
                            bindings[p2_parts[0]] = fact2.subject
                        elif p2_parts[0] != fact2.subject:
                            continue

                        if p2_parts[2].startswith('?'):
                            expected = bindings.get(p2_parts[2])
                            if expected and expected != fact2.object:
                                continue
                            bindings[p2_parts[2]] = fact2.object
                        elif p2_parts[2] != fact2.object:
                            continue

                        # Applique la conclusion
                        conclusion_parts = rule.conclusion.split()
                        if len(conclusion_parts) == 3:
                            subject = bindings.get(conclusion_parts[0], conclusion_parts[0])
                            predicate = conclusion_parts[1]
                            obj = bindings.get(conclusion_parts[2], conclusion_parts[2])

                            # Calcule la confiance
                            confidence = min(fact1.confidence, fact2.confidence) * rule.confidence_decay

                            new_fact = LogicalFact(
                                subject=subject,
                                predicate=predicate,
                                object=obj,
                                confidence=confidence,
                                source=f"inferred_by_{rule.name}"
                            )

                            inferred_facts.append(new_fact)

        return inferred_facts

    def _fact_exists(self, fact: LogicalFact) -> bool:
        """Vérifie si un fait existe déjà"""
        for existing in self.facts:
            if (existing.subject == fact.subject and
                existing.predicate == fact.predicate and
                existing.object == fact.object):
                return True
        return False

    async def find_path(self, start: str, end: str, relation: str = None) -> Optional[List[LogicalFact]]:
        """
        Trouve un chemin de raisonnement entre deux concepts

        Args:
            start: Concept de départ
            end: Concept cible
            relation: Type de relation (optionnel)

        Returns:
            Chemin de faits reliant start à end
        """
        # BFS dans le graphe de faits
        queue = [(start, [])]
        visited = {start}

        while queue:
            current, path = queue.pop(0)

            if current == end:
                return path

            # Trouve les faits partant de current
            next_facts = await self.query_facts(subject=current)

            if relation:
                next_facts = [f for f in next_facts if f.predicate == relation]

            for fact in next_facts:
                if fact.object not in visited:
                    visited.add(fact.object)
                    queue.append((fact.object, path + [fact]))

        return None

    async def answer_question(self, question: str) -> Optional[Answer]:
        """
        Répond à une question en utilisant le raisonnement

        Exemples:
        - "Est-ce que X est un Y ?"
        - "Qu'est-ce qui cause X ?"
        - "X a-t-il la propriété Y ?"
        """
        # Parse la question (simplifié)
        question_lower = question.lower().strip()

        # Question de type "est-ce que X est un Y ?"
        if "est-ce que" in question_lower or "is" in question_lower:
            # Extraction simple (à améliorer avec NLP)
            match = re.search(r"est-ce que (\w+) est (?:un |une )?(\w+)", question_lower)
            if match:
                subject, obj = match.groups()

                # Recherche directe
                direct_facts = await self.query_facts(subject=subject, predicate="is_a", object=obj)

                if direct_facts:
                    return Answer(
                        text=f"Oui, {subject} est un {obj}.",
                        confidence=direct_facts[0].confidence,
                        reasoning_path=[str(f) for f in direct_facts],
                        sources=[direct_facts[0].source]
                    )

                # Recherche par inférence
                path = await self.find_path(subject, obj, "is_a")
                if path:
                    return Answer(
                        text=f"Oui, {subject} est un {obj} (par inférence).",
                        confidence=min(f.confidence for f in path),
                        reasoning_path=[str(f) for f in path],
                        sources=["inference"]
                    )

                return Answer(
                    text=f"Je ne peux pas confirmer que {subject} est un {obj}.",
                    confidence=0.0,
                    reasoning_path=[],
                    sources=[]
                )

        return None

    def get_stats(self) -> Dict[str, Any]:
        """Retourne les statistiques du moteur de raisonnement"""
        return {
            'total_facts': len(self.facts),
            'total_rules': len(self.rules),
            **self.stats
        }

# ============================================================================
# LLM INTERFACE - INTERFACE AVEC LLM EXTERNES
# ============================================================================

class LLMProvider(str, Enum):
    """Providers LLM supportés"""
    NONE = "none"
    OPENAI = "openai"
    ANTHROPIC = "anthropic"
    DEEPSEEK = "deepseek"
    OLLAMA = "ollama"  # Modèles locaux

@dataclass
class LLMConfig:
    """Configuration LLM"""
    provider: LLMProvider = LLMProvider.NONE
    model_name: str = "gpt-4"
    api_key: Optional[str] = None
    temperature: float = 0.7
    max_tokens: int = 1024
    base_url: Optional[str] = None  # Pour Ollama ou API custom

class LLMInterface:
    """
    Interface unifiée pour interaction avec LLM

    Fonctionnalités:
    - Extraction d'entités et relations
    - Génération de texte
    - Question answering
    - Summarization
    """

    def __init__(self, config: LLMConfig, bus):
        self.config = config
        self.bus = bus
        self.available = False
        self._client = None

        self._check_availability()

        # Statistiques
        self.stats = {
            'total_calls': 0,
            'successful_calls': 0,
            'failed_calls': 0,
            'total_tokens': 0
        }

        logger.info(f"✅ LLM Interface initialized (provider: {config.provider})")

    def _check_availability(self):
        """Vérifie la disponibilité du provider LLM"""
        if self.config.provider == LLMProvider.NONE:
            logger.info("⚠️ LLM disabled (provider: none)")
            return

        if self.config.provider == LLMProvider.OPENAI:
            try:
                import openai
                if self.config.api_key:
                    self._client = openai.OpenAI(api_key=self.config.api_key)
                    self.available = True
                    logger.info("✅ OpenAI client initialized")
                else:
                    logger.warning("⚠️ OpenAI API key not provided")
            except ImportError:
                logger.warning("⚠️ OpenAI library not installed (pip install openai)")

        elif self.config.provider == LLMProvider.ANTHROPIC:
            try:
                import anthropic
                if self.config.api_key:
                    self._client = anthropic.Anthropic(api_key=self.config.api_key)
                    self.available = True
                    logger.info("✅ Anthropic client initialized")
                else:
                    logger.warning("⚠️ Anthropic API key not provided")
            except ImportError:
                logger.warning("⚠️ Anthropic library not installed (pip install anthropic)")

        elif self.config.provider == LLMProvider.OLLAMA:
            try:
                import ollama
                self._client = ollama
                self.available = True
                logger.info("✅ Ollama client initialized")
            except ImportError:
                logger.warning("⚠️ Ollama library not installed (pip install ollama)")

    async def extract_entities_and_relations(self, text: str) -> Dict[str, Any]:
        """
        Extrait des entités et relations d'un texte avec un LLM

        Returns:
            {
                'entities': [{'name': 'X', 'type': 'Person'}, ...],
                'relations': [{'subject': 'X', 'predicate': 'is_a', 'object': 'Y'}, ...]
            }
        """
        if not self.available:
            return {'entities': [], 'relations': []}

        prompt = f"""Extract entities and semantic relations from this text.

Text: "{text}"

Return a JSON with:
1. entities: list of {{name, type}}
2. relations: list of {{subject, predicate, object}}

Use predicates: is_a, part_of, has_property, causes, related_to

Example:
{{
    "entities": [
        {{"name": "Socrates", "type": "Person"}},
        {{"name": "mortal", "type": "Property"}}
    ],
    "relations": [
        {{"subject": "Socrates", "predicate": "is_a", "object": "Human"}},
        {{"subject": "Human", "predicate": "has_property", "object": "mortal"}}
    ]
}}

JSON:"""

        try:
            response = await self._call_llm(prompt)

            # Parse la réponse JSON
            json_match = re.search(r'\{.*\}', response, re.DOTALL)
            if json_match:
                result = json.loads(json_match.group())
                return result

            return {'entities': [], 'relations': []}

        except Exception as e:
            logger.error(f"❌ Entity extraction error: {e}")
            return {'entities': [], 'relations': []}

    async def answer_question(self, question: str, context: str = "") -> str:
        """Répond à une question en utilisant le contexte fourni"""
        if not self.available:
            return "LLM not available"

        prompt = f"""Based on the following context, answer the question concisely.

Context:
{context}

Question: {question}

Answer:"""

        try:
            response = await self._call_llm(prompt)
            return response.strip()
        except Exception as e:
            logger.error(f"❌ Question answering error: {e}")
            return "Unable to answer"

    async def summarize(self, text: str, max_sentences: int = 3) -> str:
        """Résume un texte"""
        if not self.available:
            return text[:200] + "..."

        prompt = f"""Summarize this text in {max_sentences} sentences maximum.

Text:
{text}

Summary:"""

        try:
            response = await self._call_llm(prompt)
            return response.strip()
        except Exception as e:
            logger.error(f"❌ Summarization error: {e}")
            return text[:200] + "..."

    async def _call_llm(self, prompt: str) -> str:
        """Appel générique au LLM"""
        self.stats['total_calls'] += 1

        try:
            if self.config.provider == LLMProvider.OPENAI:
                response = await asyncio.to_thread(
                    self._client.chat.completions.create,
                    model=self.config.model_name,
                    messages=[{"role": "user", "content": prompt}],
                    temperature=self.config.temperature,
                    max_tokens=self.config.max_tokens
                )

                self.stats['successful_calls'] += 1
                self.stats['total_tokens'] += response.usage.total_tokens

                return response.choices[0].message.content

            elif self.config.provider == LLMProvider.ANTHROPIC:
                response = await asyncio.to_thread(
                    self._client.messages.create,
                    model=self.config.model_name,
                    max_tokens=self.config.max_tokens,
                    messages=[{"role": "user", "content": prompt}]
                )

                self.stats['successful_calls'] += 1

                return response.content[0].text

            elif self.config.provider == LLMProvider.OLLAMA:
                response = await asyncio.to_thread(
                    self._client.generate,
                    model=self.config.model_name,
                    prompt=prompt
                )

                self.stats['successful_calls'] += 1

                return response['response']

            else:
                raise ValueError(f"Unsupported provider: {self.config.provider}")

        except Exception as e:
            self.stats['failed_calls'] += 1
            raise e

    def get_stats(self) -> Dict[str, Any]:
        """Retourne les statistiques d'utilisation du LLM"""
        return {
            'available': self.available,
            'provider': self.config.provider,
            **self.stats
        }

# ============================================================================
# INSTRUCTIONS POUR UTILISATION
# ============================================================================

"""
================================================================================
UTILISATION DU REASONING ENGINE
================================================================================

# 1. Initialisation
reasoning_engine = SymbolicReasoningEngine(graph, bus)

# 2. Ajouter des faits
await reasoning_engine.add_fact(LogicalFact(
    subject="Socrate",
    predicate="is_a",
    object="Humain",
    confidence=1.0
))

await reasoning_engine.add_fact(LogicalFact(
    subject="Humain",
    predicate="has_property",
    object="mortel",
    confidence=1.0
))

# 3. Déduction
new_facts = await reasoning_engine.deduce(max_depth=3)
# → Devrait inférer: "Socrate has_property mortel"

# 4. Question answering
answer = await reasoning_engine.answer_question(
    "Est-ce que Socrate est mortel ?"
)
print(answer.text)  # "Oui, Socrate est mortel"

================================================================================
UTILISATION DU LLM INTERFACE
================================================================================

# 1. Configuration
llm_config = LLMConfig(
    provider=LLMProvider.OPENAI,
    model_name="gpt-4",
    api_key="your-api-key-here",
    temperature=0.7
)

llm_interface = LLMInterface(llm_config, bus)

# 2. Extraction d'entités
text = "Albert Einstein was a theoretical physicist who developed the theory of relativity."
result = await llm_interface.extract_entities_and_relations(text)

# 3. Question answering
answer = await llm_interface.answer_question(
    question="Who developed the theory of relativity?",
    context=text
)

# 4. Summarization
summary = await llm_interface.summarize(long_text, max_sentences=3)

================================================================================
"""

#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
================================================================================
GRANDNEXUS PHASE 2 - INTELLIGENT SYSTEM
Part 2: Goal System & Metacognition
================================================================================
Nouveaux modules:
3. Goal-Oriented Agent - Planification et exécution de buts
4. Metacognitive Monitor - Auto-surveillance et conscience de soi
5. Query Interface - Interface de dialogue intelligente
================================================================================
"""


# ============================================================================
# GOAL SYSTEM - SYSTÈME DE BUTS ET PLANIFICATION
# ============================================================================

class GoalStatus(str, Enum):
    """Statut d'un but"""
    PENDING = "pending"
    ACTIVE = "active"
    COMPLETED = "completed"
    FAILED = "failed"
    ABANDONED = "abandoned"

class ActionType(str, Enum):
    """Types d'actions disponibles"""
    LEARN = "learn"              # Apprendre de nouvelles données
    REASON = "reason"            # Raisonner sur les connaissances
    QUERY = "query"              # Interroger le graphe
    EXTRACT = "extract"          # Extraire des informations
    INFER = "infer"              # Faire des inférences
    CONSOLIDATE = "consolidate"  # Consolider la mémoire

@dataclass
class Action:
    """Action atomique"""
    action_type: ActionType
    parameters: Dict[str, Any] = field(default_factory=dict)
    expected_duration: float = 1.0
    priority: int = 5

    def __str__(self):
        return f"{self.action_type}({self.parameters})"

@dataclass
class Plan:
    """Plan d'actions pour atteindre un but"""
    goal_id: str
    actions: List[Action] = field(default_factory=list)
    estimated_duration: float = 0.0
    created_at: float = field(default_factory=time.time)

    def __post_init__(self):
        self.estimated_duration = sum(a.expected_duration for a in self.actions)

@dataclass
class Goal:
    """But du système"""
    id: str
    description: str
    priority: int = 5
    deadline: Optional[float] = None
    status: GoalStatus = GoalStatus.PENDING
    progress: float = 0.0

    # Conditions de succès
    success_conditions: Dict[str, Any] = field(default_factory=dict)

    # Plan d'actions
    plan: Optional[Plan] = None

    # Historique
    created_at: float = field(default_factory=time.time)
    started_at: Optional[float] = None
    completed_at: Optional[float] = None

    # Métadonnées
    metadata: Dict[str, Any] = field(default_factory=dict)

    def is_expired(self) -> bool:
        """Vérifie si le but a dépassé sa deadline"""
        if self.deadline:
            return time.time() > self.deadline
        return False

class GoalOrientedAgent:
    """
    Agent orienté buts avec planification et exécution

    Capacités:
    - Gestion de buts multiples avec priorités
    - Planification automatique d'actions
    - Exécution asynchrone
    - Gestion des dépendances entre buts
    """

    def __init__(self, graph, memory, reasoning_engine, llm_interface, bus):
        self.graph = graph
        self.memory = memory
        self.reasoning_engine = reasoning_engine
        self.llm_interface = llm_interface
        self.bus = bus

        # Gestion des buts
        self.goals: Dict[str, Goal] = {}
        self.active_goals: List[str] = []
        self.goal_queue: deque = deque()

        # Exécution
        self.current_action: Optional[Action] = None
        self.action_history: List[Dict[str, Any]] = []

        # Statistiques
        self.stats = {
            'goals_created': 0,
            'goals_completed': 0,
            'goals_failed': 0,
            'actions_executed': 0,
            'total_planning_time': 0.0
        }

        logger.info("✅ Goal-Oriented Agent initialized")

    async def create_goal(self, description: str, priority: int = 5,
                         deadline: Optional[float] = None,
                         success_conditions: Dict[str, Any] = None) -> Goal:
        """
        Crée un nouveau but

        Args:
            description: Description du but
            priority: Priorité (1-10, 10 = le plus urgent)
            deadline: Timestamp de deadline (optionnel)
            success_conditions: Conditions de succès

        Returns:
            Le but créé
        """
        goal_id = f"goal_{int(time.time() * 1000)}"

        goal = Goal(
            id=goal_id,
            description=description,
            priority=priority,
            deadline=deadline,
            success_conditions=success_conditions or {},
            status=GoalStatus.PENDING
        )

        self.goals[goal_id] = goal
        self.goal_queue.append(goal_id)
        self.stats['goals_created'] += 1

        logger.info(f"🎯 Goal created: {description} (priority: {priority})")

        return goal

    async def plan_goal(self, goal_id: str) -> Optional[Plan]:
        """
        Crée un plan d'actions pour atteindre un but

        Args:
            goal_id: ID du but

        Returns:
            Plan d'actions
        """
        goal = self.goals.get(goal_id)
        if not goal:
            logger.error(f"❌ Goal not found: {goal_id}")
            return None

        planning_start = time.time()

        # Analyse le but pour déterminer les actions nécessaires
        actions = []

        # Extraction de mots-clés du but
        keywords = goal.description.lower().split()

        # Planification basée sur le type de but
        if any(word in keywords for word in ['learn', 'understand', 'study']):
            # But d'apprentissage
            actions.extend([
                Action(ActionType.LEARN, {'topic': goal.description}),
                Action(ActionType.EXTRACT, {'source': 'all'}),
                Action(ActionType.CONSOLIDATE, {})
            ])

        elif any(word in keywords for word in ['find', 'search', 'discover']):
            # But de recherche
            actions.extend([
                Action(ActionType.QUERY, {'query': goal.description}),
                Action(ActionType.REASON, {'depth': 2})
            ])

        elif any(word in keywords for word in ['infer', 'deduce', 'conclude']):
            # But de raisonnement
            actions.extend([
                Action(ActionType.REASON, {'depth': 3}),
                Action(ActionType.INFER, {'max_depth': 3})
            ])

        else:
            # But générique
            actions.extend([
                Action(ActionType.QUERY, {'query': goal.description}),
                Action(ActionType.REASON, {'depth': 2})
            ])

        plan = Plan(goal_id=goal_id, actions=actions)
        goal.plan = plan

        planning_time = time.time() - planning_start
        self.stats['total_planning_time'] += planning_time

        logger.info(f"📋 Plan created for '{goal.description}': {len(actions)} actions, "
                   f"estimated {plan.estimated_duration:.1f}s")

        return plan

    async def execute_goal(self, goal_id: str) -> bool:
        """
        Exécute un but (crée un plan si nécessaire et l'exécute)

        Returns:
            True si succès, False sinon
        """
        goal = self.goals.get(goal_id)
        if not goal:
            return False

        # Change le statut
        goal.status = GoalStatus.ACTIVE
        goal.started_at = time.time()
        self.active_goals.append(goal_id)

        # Crée un plan si nécessaire
        if not goal.plan:
            await self.plan_goal(goal_id)

        if not goal.plan:
            goal.status = GoalStatus.FAILED
            self.stats['goals_failed'] += 1
            return False

        # Exécute les actions du plan
        logger.info(f"▶️ Executing goal: {goal.description}")

        for i, action in enumerate(goal.plan.actions):
            try:
                success = await self._execute_action(action)

                if not success:
                    logger.warning(f"⚠️ Action failed: {action}")
                    goal.status = GoalStatus.FAILED
                    self.stats['goals_failed'] += 1
                    return False

                # Mise à jour de la progression
                goal.progress = (i + 1) / len(goal.plan.actions)

            except Exception as e:
                logger.error(f"❌ Error executing action {action}: {e}")
                goal.status = GoalStatus.FAILED
                self.stats['goals_failed'] += 1
                return False

        # Succès
        goal.status = GoalStatus.COMPLETED
        goal.completed_at = time.time()
        goal.progress = 1.0
        self.stats['goals_completed'] += 1

        if goal_id in self.active_goals:
            self.active_goals.remove(goal_id)

        duration = goal.completed_at - goal.started_at
        logger.info(f"✅ Goal completed: {goal.description} (duration: {duration:.1f}s)")

        return True

    async def _execute_action(self, action: Action) -> bool:
        """Exécute une action atomique"""
        self.current_action = action
        action_start = time.time()

        try:
            if action.action_type == ActionType.LEARN:
                # Apprentissage de nouvelles données
                topic = action.parameters.get('topic', '')
                # Simule l'apprentissage
                await asyncio.sleep(0.5)
                success = True

            elif action.action_type == ActionType.REASON:
                # Raisonnement
                depth = action.parameters.get('depth', 2)
                new_facts = await self.reasoning_engine.deduce(max_depth=depth)
                success = len(new_facts) >= 0  # Toujours succès même si 0 faits

            elif action.action_type == ActionType.QUERY:
                # Requête sur le graphe
                query = action.parameters.get('query', '')
                # Simule une requête
                await asyncio.sleep(0.3)
                success = True

            elif action.action_type == ActionType.EXTRACT:
                # Extraction d'informations
                await asyncio.sleep(0.4)
                success = True

            elif action.action_type == ActionType.INFER:
                # Inférence
                max_depth = action.parameters.get('max_depth', 3)
                new_facts = await self.reasoning_engine.deduce(max_depth=max_depth)
                success = True

            elif action.action_type == ActionType.CONSOLIDATE:
                # Consolidation mémoire
                await asyncio.sleep(0.5)
                success = True

            else:
                logger.warning(f"Unknown action type: {action.action_type}")
                success = False

            duration = time.time() - action_start

            self.action_history.append({
                'action': str(action),
                'success': success,
                'duration': duration,
                'timestamp': time.time()
            })

            self.stats['actions_executed'] += 1

            return success

        except Exception as e:
            logger.error(f"Action execution error: {e}")
            return False

        finally:
            self.current_action = None

    async def process_goal_queue(self, max_goals: int = 3):
        """
        Traite la file d'attente des buts (exécute les plus prioritaires)

        Args:
            max_goals: Nombre maximum de buts à traiter
        """
        processed = 0

        while self.goal_queue and processed < max_goals:
            goal_id = self.goal_queue.popleft()
            goal = self.goals.get(goal_id)

            if not goal or goal.status != GoalStatus.PENDING:
                continue

            # Vérifie la deadline
            if goal.is_expired():
                goal.status = GoalStatus.ABANDONED
                logger.warning(f"⏰ Goal expired: {goal.description}")
                continue

            # Exécute le but
            await self.execute_goal(goal_id)
            processed += 1

    def get_stats(self) -> Dict[str, Any]:
        """Statistiques de l'agent"""
        return {
            'total_goals': len(self.goals),
            'active_goals': len(self.active_goals),
            'pending_goals': len([g for g in self.goals.values() if g.status == GoalStatus.PENDING]),
            'completed_goals': len([g for g in self.goals.values() if g.status == GoalStatus.COMPLETED]),
            **self.stats
        }

# ============================================================================
# METACOGNITIVE MONITOR - CONSCIENCE DE SOI ET AUTO-SURVEILLANCE
# ============================================================================

class ConfidenceLevel(str, Enum):
    """Niveaux de confiance"""
    VERY_LOW = "very_low"    # < 0.3
    LOW = "low"              # 0.3-0.5
    MEDIUM = "medium"        # 0.5-0.7
    HIGH = "high"            # 0.7-0.9
    VERY_HIGH = "very_high"  # > 0.9

@dataclass
class SelfAssessment:
    """Auto-évaluation du système"""
    timestamp: float

    # Évaluation des capacités
    knowledge_completeness: float  # 0-1
    reasoning_capability: float    # 0-1
    learning_efficiency: float     # 0-1

    # Conscience de limitations
    known_gaps: List[str] = field(default_factory=list)
    uncertain_areas: List[str] = field(default_factory=list)

    # État interne
    cognitive_load: float = 0.0
    memory_pressure: float = 0.0

    # Méta-insights
    insights: List[str] = field(default_factory=list)

class MetacognitiveMonitor:
    """
    Système de métacognition et conscience de soi

    Capacités:
    - Auto-évaluation des connaissances
    - Détection des lacunes
    - Monitoring de la performance
    - Conscience des limitations
    - Auto-amélioration
    """

    def __init__(self, graph, memory, reasoning_engine, goal_agent, bus):
        self.graph = graph
        self.memory = memory
        self.reasoning_engine = reasoning_engine
        self.goal_agent = goal_agent
        self.bus = bus

        # Historique d'auto-évaluations
        self.assessments: List[SelfAssessment] = []

        # Tracking de performance
        self.performance_history: deque = deque(maxlen=100)

        # Conscience de soi
        self.self_awareness_level: float = 0.0
        self.metacognitive_state: Dict[str, Any] = {
            'knows_what_it_knows': [],
            'knows_what_it_doesnt_know': [],
            'uncertainty_domains': []
        }

        # Statistiques
        self.stats = {
            'total_assessments': 0,
            'gaps_identified': 0,
            'improvements_suggested': 0
        }

        logger.info("✅ Metacognitive Monitor initialized")

    async def perform_self_assessment(self) -> SelfAssessment:
        """
        Effectue une auto-évaluation complète du système

        Returns:
            Rapport d'auto-évaluation
        """
        logger.info("🔍 Performing self-assessment...")

        # 1. Évalue la complétude des connaissances
        graph_stats = self.graph.get_stats()
        memory_stats = self.memory.get_stats()

        total_nodes = graph_stats.get('total_nodes', 0)
        total_edges = graph_stats.get('total_edges_memory', 0)

        # Knowledge completeness basée sur la densité du graphe
        if total_nodes > 0:
            density = total_edges / (total_nodes * (total_nodes - 1)) if total_nodes > 1 else 0
            knowledge_completeness = min(1.0, density * 10)  # Scale arbitraire
        else:
            knowledge_completeness = 0.0

        # 2. Évalue les capacités de raisonnement
        reasoning_stats = self.reasoning_engine.get_stats()
        total_facts = reasoning_stats.get('total_facts', 0)
        total_inferences = reasoning_stats.get('total_inferences', 0)

        if total_facts > 0:
            reasoning_capability = min(1.0, total_inferences / total_facts)
        else:
            reasoning_capability = 0.0

        # 3. Évalue l'efficacité d'apprentissage
        if self.performance_history:
            recent_performance = list(self.performance_history)[-10:]
            learning_efficiency = np.mean(recent_performance)
        else:
            learning_efficiency = 0.5

        # 4. Identifie les lacunes
        known_gaps = await self._identify_knowledge_gaps()

        # 5. Identifie les zones d'incertitude
        uncertain_areas = await self._identify_uncertain_areas()

        # 6. État cognitif actuel
        cognitive_load = self.goal_agent.stats.get('actions_executed', 0) / 100.0
        memory_pressure = memory_stats.get('hot_cache_size', 0) / memory_stats.get('hot_cache_size', 1)

        # 7. Génère des insights
        insights = await self._generate_metacognitive_insights(
            knowledge_completeness,
            reasoning_capability,
            learning_efficiency
        )

        # Crée l'assessment
        assessment = SelfAssessment(
            timestamp=time.time(),
            knowledge_completeness=knowledge_completeness,
            reasoning_capability=reasoning_capability,
            learning_efficiency=learning_efficiency,
            known_gaps=known_gaps,
            uncertain_areas=uncertain_areas,
            cognitive_load=cognitive_load,
            memory_pressure=memory_pressure,
            insights=insights
        )

        self.assessments.append(assessment)
        self.stats['total_assessments'] += 1

        # Mise à jour du niveau de conscience de soi
        self.self_awareness_level = np.mean([
            knowledge_completeness,
            reasoning_capability,
            learning_efficiency
        ])

        logger.info(f"📊 Self-assessment completed:")
        logger.info(f"   Knowledge: {knowledge_completeness:.2%}")
        logger.info(f"   Reasoning: {reasoning_capability:.2%}")
        logger.info(f"   Learning: {learning_efficiency:.2%}")
        logger.info(f"   Self-awareness: {self.self_awareness_level:.2%}")

        return assessment

    async def _identify_knowledge_gaps(self) -> List[str]:
        """Identifie les lacunes dans les connaissances"""
        gaps = []

        # Concepts avec peu de connexions (potentiellement incomplets)
        for node, edges in self.graph.edges.items():
            if len(edges) < 2:  # Seuil arbitraire
                gaps.append(f"Concept '{node}' has few connections")

        self.stats['gaps_identified'] += len(gaps)

        return gaps[:10]  # Limite à 10 pour lisibilité

    async def _identify_uncertain_areas(self) -> List[str]:
        """Identifie les zones d'incertitude"""
        uncertain = []

        # Faits avec faible confiance
        for fact in self.reasoning_engine.facts:
            if fact.confidence < 0.5:
                uncertain.append(f"Low confidence: {fact.subject} {fact.predicate} {fact.object}")

        return uncertain[:10]

    async def _generate_metacognitive_insights(self, knowledge: float,
                                               reasoning: float,
                                               learning: float) -> List[str]:
        """Génère des insights métacognitifs"""
        insights = []

        # Analyse des forces
        if knowledge > 0.7:
            insights.append("I have a good knowledge base")
        elif knowledge < 0.3:
            insights.append("My knowledge base is limited, I need to learn more")

        if reasoning > 0.7:
            insights.append("My reasoning capabilities are strong")
        elif reasoning < 0.3:
            insights.append("I struggle with complex reasoning")

        if learning > 0.7:
            insights.append("I learn efficiently from new data")
        elif learning < 0.3:
            insights.append("My learning process needs improvement")

        # Insights sur l'équilibre
        if abs(knowledge - reasoning) > 0.3:
            if knowledge > reasoning:
                insights.append("I have knowledge but limited reasoning - need to practice inference")
            else:
                insights.append("I can reason well but lack knowledge - need to learn more")

        self.stats['improvements_suggested'] += len(insights)

        return insights

    async def can_answer_question(self, question: str) -> Tuple[bool, float, str]:
        """
        Évalue si le système peut répondre à une question

        Returns:
            (can_answer, confidence, explanation)
        """
        # Analyse simple basée sur les mots-clés présents dans le graphe
        words = question.lower().split()

        relevant_nodes = []
        for word in words:
            if word in self.graph.nodes:
                relevant_nodes.append(word)

        if not relevant_nodes:
            return (False, 0.0, "No relevant knowledge found")

        # Confiance basée sur le nombre de nœuds pertinents
        confidence = min(1.0, len(relevant_nodes) / 3)

        if confidence > 0.7:
            return (True, confidence, f"I have strong knowledge about: {', '.join(relevant_nodes)}")
        elif confidence > 0.4:
            return (True, confidence, f"I have some knowledge about: {', '.join(relevant_nodes)}")
        else:
            return (False, confidence, f"Limited knowledge about: {', '.join(relevant_nodes)}")

    async def suggest_improvements(self) -> List[str]:
        """Suggère des améliorations au système"""
        suggestions = []

        if not self.assessments:
            return ["Perform initial self-assessment"]

        latest = self.assessments[-1]

        # Suggestions basées sur l'évaluation
        if latest.knowledge_completeness < 0.5:
            suggestions.append("Increase data collection from external sources")

        if latest.reasoning_capability < 0.5:
            suggestions.append("Perform more inference cycles")

        if latest.learning_efficiency < 0.5:
            suggestions.append("Improve data processing algorithms")

        if latest.cognitive_load > 0.8:
            suggestions.append("Reduce active goals to decrease cognitive load")

        if latest.memory_pressure > 0.8:
            suggestions.append("Perform memory consolidation")

        return suggestions

    def get_stats(self) -> Dict[str, Any]:
        """Statistiques du monitor métacognitif"""
        return {
            'self_awareness_level': self.self_awareness_level,
            'total_assessments': len(self.assessments),
            **self.stats
        }

# ============================================================================
# QUERY INTERFACE - INTERFACE DE DIALOGUE
# ============================================================================

class QueryInterface:
    """
    Interface de haut niveau pour interagir avec GrandNexus

    Permet:
    - Poser des questions
    - Donner des commandes
    - Enseigner de nouveaux faits
    - Obtenir des explications
    """

    def __init__(self, reasoning_engine, llm_interface, goal_agent,
                 metacognitive_monitor, bus):
        self.reasoning_engine = reasoning_engine
        self.llm_interface = llm_interface
        self.goal_agent = goal_agent
        self.metacognitive_monitor = metacognitive_monitor
        self.bus = bus

        # Historique de conversation
        self.conversation_history: List[Dict[str, str]] = []

        # Statistiques
        self.stats = {
            'total_queries': 0,
            'questions_answered': 0,
            'commands_executed': 0,
            'facts_taught': 0
        }

        logger.info("✅ Query Interface initialized")

    async def process_query(self, query_text: str) -> Answer:
        """
        Traite une requête utilisateur

        Args:
            query_text: Texte de la requête

        Returns:
            Réponse du système
        """
        self.stats['total_queries'] += 1

        # Sauvegarde dans l'historique
        self.conversation_history.append({
            'timestamp': time.time(),
            'role': 'user',
            'content': query_text
        })

        # Détermine le type de requête
        query_text_lower = query_text.lower().strip()

        # Question
        if any(query_text_lower.startswith(q) for q in ['what', 'who', 'where', 'when', 'why', 'how', 'is', 'does', 'can']):
            answer = await self._handle_question(query_text)
            self.stats['questions_answered'] += 1

        # Commande
        elif any(query_text_lower.startswith(c) for c in ['learn', 'find', 'analyze', 'explain']):
            answer = await self._handle_command(query_text)
            self.stats['commands_executed'] += 1

        # Fait à apprendre
        elif ' is ' in query_text_lower or ' are ' in query_text_lower:
            answer = await self._handle_teaching(query_text)
            self.stats['facts_taught'] += 1

        # Autre
        else:
            answer = Answer(
                text="I'm not sure how to process this query. Can you rephrase it as a question or command?",
                confidence=0.0,
                reasoning_path=[],
                sources=[]
            )

        # Sauvegarde la réponse
        self.conversation_history.append({
            'timestamp': time.time(),
            'role': 'assistant',
            'content': answer.text
        })

        return answer

    async def _handle_question(self, question: str) -> Answer:
        """Gère une question"""
        # 1. Auto-évaluation: peut-on répondre ?
        can_answer, confidence, explanation = await self.metacognitive_monitor.can_answer_question(question)

        if not can_answer:
            return Answer(
                text=f"I cannot confidently answer this question. {explanation}",
                confidence=0.0,
                reasoning_path=[explanation],
                sources=['metacognitive_assessment']
            )

        # 2. Tentative de réponse par raisonnement symbolique
        reasoning_answer = await self.reasoning_engine.answer_question(question)

        if reasoning_answer and reasoning_answer.confidence > 0.5:
            return reasoning_answer

        # 3. Tentative avec LLM si disponible
        if self.llm_interface.available:
            # Construit le contexte depuis le graphe
            context = await self._build_context_from_graph(question)

            llm_answer = await self.llm_interface.answer_question(question, context)

            return Answer(
                text=llm_answer,
                confidence=0.7,  # Confiance arbitraire pour LLM
                reasoning_path=['llm_inference'],
                sources=['llm', 'knowledge_graph']
            )

        # 4. Réponse par défaut
        return Answer(
            text=f"I have limited information about this topic. {explanation}",
            confidence=confidence,
            reasoning_path=[explanation],
            sources=['knowledge_graph']
        )

    async def _handle_command(self, command: str) -> Answer:
        """Gère une commande"""
        command_lower = command.lower()

        if command_lower.startswith('learn'):
            # Crée un goal d'apprentissage
            goal = await self.goal_agent.create_goal(
                description=command,
                priority=7
            )
            await self.goal_agent.execute_goal(goal.id)

            return Answer(
                text=f"I will learn about this topic. Goal created: {goal.id}",
                confidence=1.0,
                reasoning_path=[f"Created learning goal: {goal.id}"],
                sources=['goal_system']
            )

        elif command_lower.startswith('find'):
            # Crée un goal de recherche
            goal = await self.goal_agent.create_goal(
                description=command,
                priority=8
            )
            await self.goal_agent.execute_goal(goal.id)

            return Answer(
                text=f"Searching... Goal created: {goal.id}",
                confidence=1.0,
                reasoning_path=[f"Created search goal: {goal.id}"],
                sources=['goal_system']
            )

        else:
            return Answer(
                text="Command not recognized. Available commands: learn, find, analyze, explain",
                confidence=0.0,
                reasoning_path=[],
                sources=[]
            )

    async def _handle_teaching(self, statement: str) -> Answer:
        """Gère l'enseignement d'un fait"""
        # Extraction simple (à améliorer avec LLM)
        if ' is a ' in statement.lower():
            parts = statement.lower().split(' is a ')
            if len(parts) == 2:
                subject = parts[0].strip()
                obj = parts[1].strip().rstrip('.')

                # Ajoute le fait
                await self.reasoning_engine.add_fact(LogicalFact(
                    subject=subject,
                    predicate="is_a",
                    object=obj,
                    confidence=1.0,
                    source="user_teaching"
                ))

                return Answer(
                    text=f"Thank you, I learned that {subject} is a {obj}.",
                    confidence=1.0,
                    reasoning_path=[f"Added fact: {subject} is_a {obj}"],
                    sources=['user']
                )

        return Answer(
            text="I'm trying to understand what you're teaching me. Can you phrase it as 'X is a Y'?",
            confidence=0.0,
            reasoning_path=[],
            sources=[]
        )

    async def _build_context_from_graph(self, query: str) -> str:
        """Construit un contexte textuel depuis le graphe pour le LLM"""
        # Extrait les mots-clés de la query
        words = query.lower().split()

        relevant_facts = []
        for word in words:
            if word in self.graph.nodes:
                # Trouve les faits liés à ce nœud
                facts = await self.reasoning_engine.query_facts(subject=word)
                relevant_facts.extend(facts)

        if not relevant_facts:
            return "No relevant information in knowledge base."

        # Construit le texte
        context_lines = [str(fact) for fact in relevant_facts[:10]]  # Limite à 10
        return "\n".join(context_lines)

    def get_stats(self) -> Dict[str, Any]:
        """Statistiques de l'interface"""
        return {
            'conversation_length': len(self.conversation_history),
            **self.stats
        }

# ============================================================================
# INSTRUCTIONS D'UTILISATION
# ============================================================================

"""
================================================================================
UTILISATION DU SYSTÈME COMPLET PHASE 2
================================================================================

# 1. Initialisation de tous les modules
reasoning_engine = SymbolicReasoningEngine(graph, bus)
llm_interface = LLMInterface(llm_config, bus)
goal_agent = GoalOrientedAgent(graph, memory, reasoning_engine, llm_interface, bus)
metacognitive_monitor = MetacognitiveMonitor(graph, memory, reasoning_engine, goal_agent, bus)
query_interface = QueryInterface(reasoning_engine, llm_interface, goal_agent, metacognitive_monitor, bus)

# 2. Interaction via l'interface
answer = await query_interface.process_query("What is artificial intelligence?")
print(answer.text)
print(f"Confidence: {answer.confidence}")

# 3. Enseigner des faits
await query_interface.process_query("Socrates is a Human")
await query_interface.process_query("Humans are mortal")

# 4. Poser des questions basées sur le raisonnement
answer = await query_interface.process_query("Is Socrates mortal?")
# → Devrait déduire via le reasoning engine

# 5. Créer des buts
goal = await goal_agent.create_goal(
    description="Learn about quantum physics",
    priority=8,
    deadline=time.time() + 3600  # 1 heure
)

# 6. Auto-évaluation
assessment = await metacognitive_monitor.perform_self_assessment()
print(f"Self-awareness level: {metacognitive_monitor.self_awareness_level}")
print(f"Insights: {assessment.insights}")

# 7. Vérifier si le système peut répondre
can_answer, confidence, explanation = await metacognitive_monitor.can_answer_question(
    "What is quantum entanglement?"
)
print(f"Can answer: {can_answer}, Confidence: {confidence}, Explanation: {explanation}")

================================================================================
"""

# ============================================================================
# ORCHESTRATEUR INTÉGRÉ PHASE 2
# ============================================================================

class IntelligentGrandNexusOrchestrator:
    """
    Orchestrateur principal intégrant tous les modules Phase 1 + Phase 2

    Architecture complète:
    - Phase 1: Data collection, Memory, Graph, Basic Cognition
    - Phase 2: Reasoning, LLM, Goals, Metacognition, Query Interface
    """

    def __init__(self, config):
        self.config = config

        # Composants Phase 1 (existants)
        self.bus = None
        self.persistence = None
        self.data_connector = None
        self.memory = None
        self.graph = None
        self.cognitive_core = None

        # Composants Phase 2 (nouveaux)
        self.reasoning_engine = None
        self.llm_interface = None
        self.goal_agent = None
        self.metacognitive_monitor = None
        self.query_interface = None

        # État
        self.running = False
        self.start_time = None
        self.cycle_count = 0

        # Métriques consolidées
        self.consolidated_stats = {}

        logger.info("🚀 Intelligent GrandNexus Orchestrator created")

    async def initialize(self):
        """Initialise tous les modules Phase 1 + Phase 2"""
        logger.info("="*80)
        logger.info("🌟 INITIALIZING INTELLIGENT GRANDNEXUS - PHASE 2")
        logger.info("="*80)

        # ===== PHASE 1 MODULES =====
        logger.info("\n📦 Phase 1: Core Infrastructure")

        # Persistence
        from pathlib import Path
        self.persistence = PersistenceManager(self.config.DB_PATH)
        logger.info("✅ Persistence manager initialized")

        # Message Bus
        self.bus = AsyncMessageBus()
        await self.bus.start()
        logger.info("✅ Message bus initialized")

        # Data Connector
        self.data_connector = RealDataConnector()
        logger.info("✅ Data connector initialized")

        # Memory
        self.memory = RealHierarchicalMemory(self.bus, self.persistence)
        logger.info("✅ Memory system initialized")

        # Graph
        self.graph = RealConceptualGraph(self.bus, self.persistence)
        logger.info("✅ Conceptual graph initialized")

        # Cognitive Core
        self.cognitive_core = RealCognitiveCore(self.bus)
        logger.info("✅ Cognitive core initialized")

        # ===== PHASE 2 MODULES =====
        logger.info("\n🧠 Phase 2: Intelligence Layer")

        # Reasoning Engine
        self.reasoning_engine = SymbolicReasoningEngine(self.graph, self.bus)
        logger.info("✅ Reasoning engine initialized")

        # LLM Interface
        llm_config = LLMConfig(
            provider=getattr(self.config, 'LLM_PROVIDER', LLMProvider.NONE),
            model_name=getattr(self.config, 'LLM_MODEL', 'gpt-4'),
            api_key=getattr(self.config, 'LLM_API_KEY', None)
        )
        self.llm_interface = LLMInterface(llm_config, self.bus)
        logger.info("✅ LLM interface initialized")

        # Goal Agent
        self.goal_agent = GoalOrientedAgent(
            self.graph,
            self.memory,
            self.reasoning_engine,
            self.llm_interface,
            self.bus
        )
        logger.info("✅ Goal-oriented agent initialized")

        # Metacognitive Monitor
        self.metacognitive_monitor = MetacognitiveMonitor(
            self.graph,
            self.memory,
            self.reasoning_engine,
            self.goal_agent,
            self.bus
        )
        logger.info("✅ Metacognitive monitor initialized")

        # Query Interface
        self.query_interface = QueryInterface(
            self.reasoning_engine,
            self.llm_interface,
            self.goal_agent,
            self.metacognitive_monitor,
            self.bus
        )
        logger.info("✅ Query interface initialized")

        # Initialisation des concepts de base avec relations logiques
        await self._initialize_knowledge_base()

        logger.info("\n" + "="*80)
        logger.info("🎯 INTELLIGENT GRANDNEXUS READY!")
        logger.info("="*80)

    async def _initialize_knowledge_base(self):
        """Initialise une base de connaissances avec faits logiques"""
        logger.info("📚 Initializing knowledge base with logical facts...")

        # Concepts de base dans le graphe
        base_concepts = [
            ('system_self', {'type': 'identity', 'importance': 1.0}),
            ('knowledge', {'type': 'abstract', 'importance': 0.9}),
            ('learning', {'type': 'process', 'importance': 0.9}),
            ('reasoning', {'type': 'capability', 'importance': 0.9}),
            ('intelligence', {'type': 'abstract', 'importance': 0.9})
        ]

        for name, attrs in base_concepts:
            await self.graph.add_concept(name, attrs)
            await self.memory.store(
                f"concept_{name}",
                attrs,
                importance=attrs.get('importance', 0.5),
                semantic_tags=['concept', 'base', attrs.get('type')]
            )

        # Relations dans le graphe
        await self.graph.add_relation('system_self', 'knowledge', 'possesses', 0.9)
        await self.graph.add_relation('system_self', 'learning', 'performs', 0.9)
        await self.graph.add_relation('system_self', 'reasoning', 'has_capability', 0.9)
        await self.graph.add_relation('learning', 'knowledge', 'produces', 0.8)

        # Faits logiques dans le reasoning engine
        basic_facts = [
            LogicalFact('system_self', 'is_a', 'intelligent_system', 1.0, 'system'),
            LogicalFact('intelligent_system', 'has_property', 'can_reason', 1.0, 'system'),
            LogicalFact('intelligent_system', 'has_property', 'can_learn', 1.0, 'system'),
            LogicalFact('learning', 'causes', 'knowledge_growth', 0.9, 'system'),
            LogicalFact('reasoning', 'requires', 'knowledge', 0.95, 'system')
        ]

        for fact in basic_facts:
            await self.reasoning_engine.add_fact(fact)

        # Déduction initiale
        new_facts = await self.reasoning_engine.deduce(max_depth=2)

        logger.info(f"✅ Knowledge base initialized: {len(base_concepts)} concepts, "
                   f"{len(basic_facts)} facts, {len(new_facts)} inferred")

    async def run(self, duration_hours: float = 3.0, enable_interactive: bool = False):
        """
        Exécution principale du système intelligent

        Args:
            duration_hours: Durée d'exécution
            enable_interactive: Active le mode interactif (REPL)
        """
        self.running = True
        self.start_time = time.time()
        end_time = self.start_time + (duration_hours * 3600)

        logger.info(f"\n🏃 Starting Intelligent GrandNexus for {duration_hours} hours...")

        # Tâche d'exécution principale
        main_task = asyncio.create_task(self._main_loop(end_time))

        # Tâche interactive optionnelle
        if enable_interactive:
            interactive_task = asyncio.create_task(self._interactive_loop())
        else:
            interactive_task = None

        try:
            if interactive_task:
                await asyncio.gather(main_task, interactive_task)
            else:
                await main_task

        except KeyboardInterrupt:
            logger.info("\n⏹️ Interrupted by user")
        except Exception as e:
            logger.error(f"💥 System error: {e}", exc_info=True)
        finally:
            await self.shutdown()

    async def _main_loop(self, end_time: float):
        """Boucle principale d'exécution"""
        last_data_fetch = time.time()
        last_checkpoint = time.time()
        last_self_assessment = time.time()
        last_goal_processing = time.time()

        while self.running and time.time() < end_time:
            cycle_start = time.time()

            # 1. COLLECTE DE DONNÉES (toutes les 5 minutes)
            if time.time() - last_data_fetch >= 300:
                await self._fetch_real_data()
                last_data_fetch = time.time()

            # 2. TRAITEMENT DES DONNÉES
            await self._process_unprocessed_data()

            # 3. CYCLE COGNITIF
            graph_stats = self.graph.get_stats()
            memory_stats = self.memory.get_stats()
            await self.cognitive_core.cognitive_cycle(graph_stats, memory_stats)

            # 4. RAISONNEMENT (tous les 10 cycles)
            if self.cycle_count % 10 == 0:
                new_facts = await self.reasoning_engine.deduce(max_depth=2)
                if new_facts:
                    logger.info(f"🔍 Inferred {len(new_facts)} new facts")

            # 5. AUTO-ÉVALUATION (toutes les 30 minutes)
            if time.time() - last_self_assessment >= 1800:
                assessment = await self.metacognitive_monitor.perform_self_assessment()

                # Suggestions d'amélioration
                suggestions = await self.metacognitive_monitor.suggest_improvements()
                if suggestions:
                    logger.info(f"💡 Improvement suggestions: {suggestions}")

                last_self_assessment = time.time()

            # 6. TRAITEMENT DES BUTS (tous les 20 cycles)
            if time.time() - last_goal_processing >= 120:  # 2 minutes
                await self.goal_agent.process_goal_queue(max_goals=2)
                last_goal_processing = time.time()

            # 7. CONSOLIDATION MÉMOIRE
            if self.cycle_count % 50 == 0:
                await self._memory_consolidation()

            # 8. CHECKPOINT (toutes les 10 minutes)
            if time.time() - last_checkpoint >= 600:
                await self._save_checkpoint()
                last_checkpoint = time.time()

            # 9. AFFICHAGE PÉRIODIQUE
            if self.cycle_count % 30 == 0:
                self._print_status()

            self.cycle_count += 1

            # Pause entre cycles
            await asyncio.sleep(self.config.CYCLE_INTERVAL_SECONDS)

    async def _interactive_loop(self):
        """Boucle interactive pour dialogue avec le système"""
        logger.info("\n💬 Interactive mode enabled. Type 'exit' to quit.")
        logger.info("Examples:")
        logger.info("  - What is artificial intelligence?")
        logger.info("  - Learn about quantum physics")
        logger.info("  - Socrates is a Human")

        while self.running:
            try:
                # Simule une interface interactive (à adapter selon environnement)
                await asyncio.sleep(5)

                # Dans un environnement réel, on utiliserait input() dans un thread séparé
                # Ici c'est un placeholder

            except Exception as e:
                logger.error(f"Interactive loop error: {e}")

    async def _fetch_real_data(self):
        """Collecte des données réelles"""
        logger.info("📡 Fetching real-world data...")

        # Wikipedia
        if self.config.ENABLE_WIKIPEDIA and self.data_connector.wikipedia_available:
            topics = ["Artificial Intelligence", "Machine Learning", "Neural Networks",
                     "Cognitive Science", "Knowledge Graphs"]
            topic = topics[self.cycle_count % len(topics)]

            data_point = await self.data_connector.fetch_wikipedia_summary(topic)
            if data_point:
                self.persistence.store_real_data(data_point)

                # Extraction sémantique avec LLM si disponible
                if self.llm_interface.available:
                    extraction = await self.llm_interface.extract_entities_and_relations(
                        data_point.content
                    )

                    # Ajoute les entités et relations extraites
                    for entity in extraction.get('entities', [])[:5]:
                        entity_name = entity.get('name', '').lower()
                        if entity_name:
                            await self.graph.add_concept(entity_name, {
                                'type': entity.get('type', 'unknown'),
                                'source': 'llm_extraction'
                            })

                    for relation in extraction.get('relations', [])[:5]:
                        subject = relation.get('subject', '').lower()
                        predicate = relation.get('predicate', 'related_to')
                        obj = relation.get('object', '').lower()

                        if subject and obj:
                            # Ajoute au graphe
                            if subject in self.graph.nodes and obj in self.graph.nodes:
                                await self.graph.add_relation(subject, obj, predicate, 0.7)

                            # Ajoute au reasoning engine comme fait logique
                            await self.reasoning_engine.add_fact(LogicalFact(
                                subject=subject,
                                predicate=predicate,
                                object=obj,
                                confidence=0.7,
                                source='llm_extraction'
                            ))

                logger.info(f"✅ Fetched and processed: {topic}")

        # RSS Feeds
        if self.config.ENABLE_RSS_FEEDS and self.data_connector.rss_available:
            from pathlib import Path
            # Import DEFAULT_RSS_FEEDS from the data connector module
            DEFAULT_RSS_FEEDS = [
                "http://rss.arxiv.org/rss/cs.AI",
                "https://news.ycombinator.com/rss"
            ]

            data_points = await self.data_connector.fetch_rss_feeds(DEFAULT_RSS_FEEDS)
            for dp in data_points:
                self.persistence.store_real_data(dp)

            if data_points:
                logger.info(f"✅ Fetched {len(data_points)} RSS articles")

    async def _process_unprocessed_data(self):
        """Traite les données en attente"""
        data_points = self.persistence.get_unprocessed_data(limit=5)

        for dp in data_points:
            await self._process_data_point(dp)
            self.persistence.mark_as_processed(dp.hash)
            self.cognitive_core.stats['data_processed'] += 1

    async def _process_data_point(self, data_point):
        """Traite un point de données"""
        # Extraction de concepts
        words = data_point.content.split()
        important_words = [w for w in words if len(w) > 5][:10]

        created_concepts = []

        for word in important_words:
            word_clean = word.strip('.,!?;:()[]{}"\'"').lower()

            if word_clean.startswith('http') or '<' in word_clean or '>' in word_clean:
                continue

            if word_clean not in self.graph.nodes:
                await self.graph.add_concept(word_clean, {
                    'type': 'extracted',
                    'source': data_point.source,
                    'frequency': 1
                })

                await self.memory.store(
                    f"concept_{word_clean}",
                    {
                        'word': word_clean,
                        'first_seen': data_point.timestamp,
                        'source': data_point.source
                    },
                    importance=0.6,
                    semantic_tags=['concept', 'extracted', data_point.source]
                )

                created_concepts.append(word_clean)
                self.cognitive_core.stats['concepts_learned'] += 1
            else:
                created_concepts.append(word_clean)

        # Création de relations
        for i, word_clean in enumerate(created_concepts):
            if i > 0:
                prev_word = created_concepts[i - 1]
                await self.graph.add_relation(prev_word, word_clean, 'co_occurs', 0.6)

            if i < len(created_concepts) - 1:
                next_word = created_concepts[i + 1]
                await self.graph.add_relation(word_clean, next_word, 'precedes', 0.5)

    async def _memory_consolidation(self):
        """Consolidation de la mémoire"""
        logger.info("🧠 Memory consolidation...")

        promoted_count = 0
        for key, count in self.memory.access_counts.items():
            if count > 10:
                current_importance = self.memory.importance_scores.get(key, 0.5)
                self.memory.importance_scores[key] = min(1.0, current_importance + 0.05)
                promoted_count += 1

        if promoted_count > 0:
            logger.info(f"✅ Promoted {promoted_count} items in memory")

    async def _save_checkpoint(self):
        """Sauvegarde un checkpoint"""
        checkpoint_file = self.config.CHECKPOINT_DIR / f"checkpoint_{int(time.time())}.json"

        checkpoint_data = {
            'timestamp': time.time(),
            'cycle_count': self.cycle_count,
            'uptime_seconds': time.time() - self.start_time,
            'phase1_stats': {
                'memory': self.memory.get_stats(),
                'graph': self.graph.get_stats(),
                'cognitive': self.cognitive_core.get_stats(),
                'db': self.persistence.get_statistics()
            },
            'phase2_stats': {
                'reasoning': self.reasoning_engine.get_stats(),
                'llm': self.llm_interface.get_stats(),
                'goals': self.goal_agent.get_stats(),
                'metacognition': self.metacognitive_monitor.get_stats(),
                'query': self.query_interface.get_stats()
            }
        }

        with open(checkpoint_file, 'w') as f:
            json.dump(checkpoint_data, f, indent=2)

        logger.info(f"💾 Checkpoint saved: {checkpoint_file.name}")

    def _print_status(self):
        """Affiche le statut système complet"""
        uptime = time.time() - self.start_time
        hours, remainder = divmod(uptime, 3600)
        minutes, seconds = divmod(remainder, 60)

        # Phase 1 stats
        memory_stats = self.memory.get_stats()
        graph_stats = self.graph.get_stats()
        cognitive_stats = self.cognitive_core.get_stats()
        db_stats = self.persistence.get_statistics()

        # Phase 2 stats
        reasoning_stats = self.reasoning_engine.get_stats()
        goal_stats = self.goal_agent.get_stats()
        meta_stats = self.metacognitive_monitor.get_stats()
        query_stats = self.query_interface.get_stats()

        logger.info(f"""
{'='*80}
📊 INTELLIGENT GRANDNEXUS STATUS - Cycle #{self.cycle_count}
⏱️  Uptime: {int(hours)}h {int(minutes)}m {int(seconds)}s

🧠 PHASE 1 - COGNITIVE CORE:
   Consciousness: {cognitive_stats['consciousness_level']:.3f}
   Knowledge Density: {cognitive_stats['knowledge_density']:.3f}
   Learning Rate: {cognitive_stats['learning_rate']:.4f}
   Data Processed: {cognitive_stats['data_processed']}
   Concepts Learned: {cognitive_stats['concepts_learned']}

💾 MEMORY:
   Hot: {memory_stats['hot_cache_size']} | Compressed: {memory_stats['compressed_size']}
   Stores: {memory_stats['stores']} | Retrievals: {memory_stats['retrievals']}

🕸️  KNOWLEDGE GRAPH:
   Nodes: {graph_stats['total_nodes']}
   Edges: {graph_stats['total_edges_persisted']}

🔬 PHASE 2 - REASONING:
   Total Facts: {reasoning_stats['total_facts']}
   Total Inferences: {reasoning_stats['total_inferences']}
   Deductions: {reasoning_stats['deductions']}

🎯 GOALS:
   Total: {goal_stats['total_goals']}
   Active: {goal_stats['active_goals']}
   Completed: {goal_stats['completed_goals']}
   Actions Executed: {goal_stats['actions_executed']}

🪞 METACOGNITION:
   Self-Awareness: {meta_stats['self_awareness_level']:.3f}
   Assessments: {meta_stats['total_assessments']}
   Gaps Identified: {meta_stats['gaps_identified']}

💬 QUERIES:
   Total: {query_stats['total_queries']}
   Questions: {query_stats['questions_answered']}
   Commands: {query_stats['commands_executed']}

💿 DATABASE:
   Concepts: {db_stats['total_concepts']}
   Relations: {db_stats['total_relations']}
   Data Points: {db_stats['total_data_points']}
{'='*80}
        """)

    async def shutdown(self):
        """Arrêt propre du système"""
        logger.info("🛑 Shutting down Intelligent GrandNexus System...")

        await self._save_final_report()

        if self.bus:
            await self.bus.stop()

        logger.info("✅ Shutdown complete")

    async def _save_final_report(self):
        """Génère et sauvegarde le rapport final"""
        final_report = {
            'execution_summary': {
                'start_time': datetime.fromtimestamp(self.start_time).isoformat(),
                'end_time': datetime.now().isoformat(),
                'total_runtime_hours': (time.time() - self.start_time) / 3600,
                'total_cycles': self.cycle_count
            },
            'phase1_metrics': {
                'cognitive': self.cognitive_core.get_stats(),
                'memory': self.memory.get_stats(),
                'graph': self.graph.get_stats(),
                'database': self.persistence.get_statistics()
            },
            'phase2_metrics': {
                'reasoning': self.reasoning_engine.get_stats(),
                'llm': self.llm_interface.get_stats(),
                'goals': self.goal_agent.get_stats(),
                'metacognition': self.metacognitive_monitor.get_stats(),
                'query': self.query_interface.get_stats()
            }
        }

        report_file = self.config.DATA_DIR / f"final_report_phase2_{datetime.now():%Y%m%d_%H%M%S}.json"

        with open(report_file, 'w') as f:
            json.dump(final_report, f, indent=2)

        logger.info(f"📄 Final report saved: {report_file.name}")

        # Affichage console
        print("\n" + "="*80)
        print("📊 FINAL EXECUTION REPORT - PHASE 2")
        print("="*80)
        print(f"Runtime: {final_report['execution_summary']['total_runtime_hours']:.2f} hours")
        print(f"Total Cycles: {final_report['execution_summary']['total_cycles']}")
        print(f"\n🔬 INTELLIGENCE METRICS:")
        print(f"   Total Facts: {final_report['phase2_metrics']['reasoning']['total_facts']}")
        print(f"   Inferences Made: {final_report['phase2_metrics']['reasoning']['total_inferences']}")
        print(f"   Goals Completed: {final_report['phase2_metrics']['goals']['completed_goals']}")
        print(f"   Self-Awareness: {final_report['phase2_metrics']['metacognition']['self_awareness_level']:.3f}")
        print(f"   Questions Answered: {final_report['phase2_metrics']['query']['questions_answered']}")
        print("="*80)

# ============================================================================
# FONCTIONS D'ENTRÉE POUR PHASE 2
# ============================================================================

async def run_intelligent_grandnexus(duration_hours: float = 3.0,
                                     enable_wikipedia: bool = True,
                                     enable_rss: bool = True,
                                     enable_llm: bool = False,
                                     llm_provider: str = "none",
                                     llm_api_key: str = None,
                                     enable_interactive: bool = False):
    """
    Lance GrandNexus Phase 2 - Système Intelligent Complet

    Args:
        duration_hours: Durée d'exécution
        enable_wikipedia: Activer Wikipedia
        enable_rss: Activer RSS feeds
        enable_llm: Activer l'interface LLM
        llm_provider: Provider LLM (openai, anthropic, ollama)
        llm_api_key: Clé API pour le LLM
        enable_interactive: Mode interactif

    Returns:
        IntelligentGrandNexusOrchestrator instance
    """

    # Configuration
    config = RealSystemConfig()
    config.DEFAULT_RUNTIME_HOURS = duration_hours
    config.ENABLE_WIKIPEDIA = enable_wikipedia
    config.ENABLE_RSS_FEEDS = enable_rss

    # Configuration LLM
    if enable_llm:
        config.LLM_PROVIDER = llm_provider
        config.LLM_API_KEY = llm_api_key
    else:
        config.LLM_PROVIDER = "none"

    # Installation des dépendances si nécessaire
    if enable_wikipedia:
        try:
            import wikipediaapi
        except ImportError:
            print("📦 Installing wikipedia-api...")
            import subprocess
            subprocess.check_call(['pip', 'install', '-q', 'wikipedia-api'])

    if enable_rss:
        try:
            import feedparser
        except ImportError:
            print("📦 Installing feedparser...")
            import subprocess
            subprocess.check_call(['pip', 'install', '-q', 'feedparser'])

    if enable_llm and llm_provider == "openai":
        try:
            import openai
        except ImportError:
            print("📦 Installing openai...")
            import subprocess
            subprocess.check_call(['pip', 'install', '-q', 'openai'])

    # Création et lancement
    orchestrator = IntelligentGrandNexusOrchestrator(config)
    await orchestrator.initialize()
    await orchestrator.run(duration_hours, enable_interactive)

    return orchestrator

# ============================================================================
# TESTS ET EXEMPLES
# ============================================================================

async def test_reasoning_capabilities():
    """Test des capacités de raisonnement"""
    print("\n🧪 Testing Reasoning Capabilities...")

    config = RealSystemConfig()
    config.LLM_PROVIDER = "none"

    orchestrator = IntelligentGrandNexusOrchestrator(config)
    await orchestrator.initialize()

    # Test 1: Enseigner des faits
    print("\n1️⃣ Teaching facts...")
    await orchestrator.query_interface.process_query("Socrates is a Human")
    await orchestrator.query_interface.process_query("Humans are mortal")

    # Déduction
    new_facts = await orchestrator.reasoning_engine.deduce(max_depth=3)
    print(f"   ✅ Inferred {len(new_facts)} new facts")
    for fact in new_facts:
        print(f"      {fact}")

    # Test 2: Poser une question
    print("\n2️⃣ Asking question...")
    answer = await orchestrator.query_interface.process_query("Is Socrates mortal?")
    print(f"   Answer: {answer.text}")
    print(f"   Confidence: {answer.confidence}")
    print(f"   Reasoning: {answer.reasoning_path}")

    # Test 3: Auto-évaluation
    print("\n3️⃣ Self-assessment...")
    assessment = await orchestrator.metacognitive_monitor.perform_self_assessment()
    print(f"   Knowledge completeness: {assessment.knowledge_completeness:.2%}")
    print(f"   Reasoning capability: {assessment.reasoning_capability:.2%}")
    print(f"   Insights: {assessment.insights}")

    return orchestrator

async def test_goal_system():
    """Test du système de buts"""
    print("\n🎯 Testing Goal System...")

    config = RealSystemConfig()
    config.LLM_PROVIDER = "none"

    orchestrator = IntelligentGrandNexusOrchestrator(config)
    await orchestrator.initialize()

    # Créer des buts
    print("\n1️⃣ Creating goals...")
    goal1 = await orchestrator.goal_agent.create_goal(
        description="Learn about artificial intelligence",
        priority=8
    )
    print(f"   ✅ Goal created: {goal1.description}")

    goal2 = await orchestrator.goal_agent.create_goal(
        description="Find information about machine learning",
        priority=7
    )
    print(f"   ✅ Goal created: {goal2.description}")

    # Exécuter les buts
    print("\n2️⃣ Executing goals...")
    success1 = await orchestrator.goal_agent.execute_goal(goal1.id)
    print(f"   Goal 1: {'✅ Success' if success1 else '❌ Failed'}")

    success2 = await orchestrator.goal_agent.execute_goal(goal2.id)
    print(f"   Goal 2: {'✅ Success' if success2 else '❌ Failed'}")

    # Statistiques
    print("\n3️⃣ Goal statistics...")
    stats = orchestrator.goal_agent.get_stats()
    print(f"   Total goals: {stats['total_goals']}")
    print(f"   Completed: {stats['completed_goals']}")
    print(f"   Actions executed: {stats['actions_executed']}")

    return orchestrator

async def test_metacognition():
    """Test de la métacognition"""
    print("\n🪞 Testing Metacognition...")

    config = RealSystemConfig()
    config.LLM_PROVIDER = "none"

    orchestrator = IntelligentGrandNexusOrchestrator(config)
    await orchestrator.initialize()

    # Ajouter des connaissances
    print("\n1️⃣ Adding knowledge...")
    await orchestrator.query_interface.process_query("Einstein is a Physicist")
    await orchestrator.query_interface.process_query("Physicists study physics")
    await orchestrator.query_interface.process_query("Physics is a Science")

    # Test de conscience de capacité
    print("\n2️⃣ Testing self-awareness...")

    questions = [
        "What is physics?",
        "Who is Einstein?",
        "What is quantum mechanics?",  # Ne sait probablement pas
    ]

    for question in questions:
        can_answer, confidence, explanation = await orchestrator.metacognitive_monitor.can_answer_question(question)
        print(f"\n   Q: {question}")
        print(f"   Can answer: {can_answer}")
        print(f"   Confidence: {confidence:.2%}")
        print(f"   Explanation: {explanation}")

    # Auto-évaluation complète
    print("\n3️⃣ Full self-assessment...")
    assessment = await orchestrator.metacognitive_monitor.perform_self_assessment()
    print(f"   Self-awareness level: {orchestrator.metacognitive_monitor.self_awareness_level:.2%}")
    print(f"   Known gaps: {len(assessment.known_gaps)}")
    print(f"   Uncertain areas: {len(assessment.uncertain_areas)}")
    print(f"   Insights:")
    for insight in assessment.insights:
        print(f"      - {insight}")

    # Suggestions d'amélioration
    print("\n4️⃣ Improvement suggestions...")
    suggestions = await orchestrator.metacognitive_monitor.suggest_improvements()
    for suggestion in suggestions:
        print(f"      💡 {suggestion}")

    return orchestrator

async def test_full_conversation():
    """Test d'une conversation complète"""
    print("\n💬 Testing Full Conversation...")

    config = RealSystemConfig()
    config.LLM_PROVIDER = "none"

    orchestrator = IntelligentGrandNexusOrchestrator(config)
    await orchestrator.initialize()

    conversation = [
        "A cat is an Animal",
        "Animals are living beings",
        "Living beings need food",
        "Is a cat a living being?",
        "Does a cat need food?",
        "What do you know about cats?",
    ]

    print("\n📝 Conversation:")
    for i, query in enumerate(conversation, 1):
        print(f"\n{i}. User: {query}")
        answer = await orchestrator.query_interface.process_query(query)
        print(f"   AI: {answer.text}")
        if answer.confidence < 1.0:
            print(f"   Confidence: {answer.confidence:.2%}")

    # Statistiques finales
    print("\n\n📊 Conversation Statistics:")
    stats = orchestrator.query_interface.get_stats()
    print(f"   Total queries: {stats['total_queries']}")
    print(f"   Questions answered: {stats['questions_answered']}")
    print(f"   Facts taught: {stats['facts_taught']}")

    return orchestrator

async def benchmark_system():
    """Benchmark complet du système"""
    print("\n⚡ SYSTEM BENCHMARK")
    print("="*80)

    config = RealSystemConfig()
    config.LLM_PROVIDER = "none"

    orchestrator = IntelligentGrandNexusOrchestrator(config)

    # 1. Temps d'initialisation
    print("\n1️⃣ Initialization...")
    init_start = time.time()
    await orchestrator.initialize()
    init_time = time.time() - init_start
    print(f"   ✅ Initialization time: {init_time:.2f}s")

    # 2. Vitesse d'apprentissage
    print("\n2️⃣ Learning speed...")
    facts_to_teach = [
        "A dog is an Animal",
        "A cat is an Animal",
        "Animals have cells",
        "Cells contain DNA",
        "Dogs are mammals",
        "Mammals are warm-blooded",
        "Birds can fly",
        "Eagles are birds",
        "Fish live in water",
        "Sharks are fish"
    ]

    learn_start = time.time()
    for fact in facts_to_teach:
        await orchestrator.query_interface.process_query(fact)
    learn_time = time.time() - learn_start

    print(f"   ✅ Learned {len(facts_to_teach)} facts in {learn_time:.2f}s")
    print(f"   Average: {learn_time/len(facts_to_teach):.3f}s per fact")

    # 3. Vitesse de raisonnement
    print("\n3️⃣ Reasoning speed...")
    reason_start = time.time()
    new_facts = await orchestrator.reasoning_engine.deduce(max_depth=3)
    reason_time = time.time() - reason_start

    print(f"   ✅ Inferred {len(new_facts)} facts in {reason_time:.2f}s")
    print(f"   Average: {reason_time/max(len(new_facts), 1):.3f}s per inference")

    # 4. Vitesse de requête
    print("\n4️⃣ Query speed...")
    queries = [
        "Is a dog an Animal?",
        "Are mammals warm-blooded?",
        "Can eagles fly?",
        "Do fish live in water?",
        "Is a shark a mammal?"
    ]

    query_times = []
    for query in queries:
        query_start = time.time()
        answer = await orchestrator.query_interface.process_query(query)
        query_time = time.time() - query_start
        query_times.append(query_time)
        print(f"   Q: {query}")
        print(f"   A: {answer.text} ({query_time:.3f}s)")

    avg_query_time = np.mean(query_times)
    print(f"\n   ✅ Average query time: {avg_query_time:.3f}s")

    # 5. Capacité mémoire
    print("\n5️⃣ Memory capacity...")
    memory_stats = orchestrator.memory.get_stats()
    print(f"   Hot cache: {memory_stats['hot_cache_size']} items")
    print(f"   Total stores: {memory_stats['stores']}")
    print(f"   Total retrievals: {memory_stats['retrievals']}")

    # 6. Graphe de connaissances
    print("\n6️⃣ Knowledge graph size...")
    graph_stats = orchestrator.graph.get_stats()
    print(f"   Nodes: {graph_stats['total_nodes']}")
    print(f"   Edges: {graph_stats['total_edges_persisted']}")
    print(f"   Density: {graph_stats['total_edges_persisted'] / max(graph_stats['total_nodes'], 1):.2f}")

    # 7. Score global
    print("\n" + "="*80)
    print("📊 BENCHMARK SUMMARY")
    print("="*80)

    # Calcul du score (arbitraire mais indicatif)
    init_score = max(0, 100 - init_time * 10)  # Pénalité si > 10s
    learn_score = max(0, 100 - (learn_time / len(facts_to_teach)) * 100)  # Pénalité si > 1s/fact
    reason_score = min(100, len(new_facts) * 5)  # Bonus pour inférences
    query_score = max(0, 100 - avg_query_time * 50)  # Pénalité si > 2s
    knowledge_score = min(100, graph_stats['total_nodes'] * 2)  # Bonus pour connaissances

    total_score = (init_score + learn_score + reason_score + query_score + knowledge_score) / 5

    print(f"\nInitialization:     {init_score:.1f}/100")
    print(f"Learning Speed:     {learn_score:.1f}/100")
    print(f"Reasoning Power:    {reason_score:.1f}/100")
    print(f"Query Speed:        {query_score:.1f}/100")
    print(f"Knowledge Base:     {knowledge_score:.1f}/100")
    print(f"\n{'='*80}")
    print(f"TOTAL SCORE:        {total_score:.1f}/100")
    print(f"{'='*80}")

    # Interprétation
    if total_score >= 80:
        grade = "🌟 EXCELLENT"
    elif total_score >= 60:
        grade = "✅ GOOD"
    elif total_score >= 40:
        grade = "⚠️  AVERAGE"
    else:
        grade = "❌ NEEDS IMPROVEMENT"

    print(f"\nPERFORMANCE GRADE: {grade}")

    return orchestrator

# ============================================================================
# FONCTIONS PRATIQUES POUR COLAB
# ============================================================================

async def quick_demo():
    """Démo rapide de toutes les capacités"""
    print("\n🎬 GRANDNEXUS PHASE 2 - QUICK DEMO")
    print("="*80)

    # Lancement rapide
    orchestrator = await run_intelligent_grandnexus(
        duration_hours=0.1,  # 6 minutes
        enable_wikipedia=False,
        enable_rss=False,
        enable_llm=False
    )

    return orchestrator

async def run_with_llm(provider="openai", api_key=None):
    """Lance avec un LLM activé"""
    print(f"\n🤖 GRANDNEXUS PHASE 2 - WITH {provider.upper()} LLM")
    print("="*80)

    if not api_key:
        print("⚠️  Warning: No API key provided. LLM will be disabled.")
        provider = "none"

    orchestrator = await run_intelligent_grandnexus(
        duration_hours=1.0,
        enable_wikipedia=True,
        enable_rss=True,
        enable_llm=(api_key is not None),
        llm_provider=provider,
        llm_api_key=api_key
    )

    return orchestrator

async def interactive_session():
    """Session interactive avec l'utilisateur"""
    print("\n💬 GRANDNEXUS PHASE 2 - INTERACTIVE SESSION")
    print("="*80)
    print("\nInitializing system...")

    config = RealSystemConfig()
    config.LLM_PROVIDER = "none"

    orchestrator = IntelligentGrandNexusOrchestrator(config)
    await orchestrator.initialize()

    print("\n✅ System ready! You can now interact with GrandNexus.")
    print("\nExamples:")
    print("  - Socrates is a Human")
    print("  - Humans are mortal")
    print("  - Is Socrates mortal?")
    print("  - What do you know about Socrates?")
    print("\nType 'exit' to quit, 'stats' for statistics, 'assess' for self-assessment.\n")

    while True:
        try:
            # Dans Colab, utiliser input() directement
            user_input = input("You: ").strip()

            if not user_input:
                continue

            if user_input.lower() == 'exit':
                print("\n👋 Goodbye!")
                break

            if user_input.lower() == 'stats':
                print("\n📊 System Statistics:")
                print(f"   Knowledge: {orchestrator.graph.get_stats()['total_nodes']} concepts")
                print(f"   Relations: {orchestrator.graph.get_stats()['total_edges_persisted']}")
                print(f"   Facts: {orchestrator.reasoning_engine.get_stats()['total_facts']}")
                print(f"   Queries: {orchestrator.query_interface.get_stats()['total_queries']}")
                print(f"   Self-awareness: {orchestrator.metacognitive_monitor.self_awareness_level:.2%}\n")
                continue

            if user_input.lower() == 'assess':
                print("\n🔍 Performing self-assessment...")
                assessment = await orchestrator.metacognitive_monitor.perform_self_assessment()
                print(f"   Knowledge: {assessment.knowledge_completeness:.2%}")
                print(f"   Reasoning: {assessment.reasoning_capability:.2%}")
                print(f"   Learning: {assessment.learning_efficiency:.2%}")
                print(f"   Insights: {assessment.insights}\n")
                continue

            # Traiter la requête
            answer = await orchestrator.query_interface.process_query(user_input)
            print(f"\nAI: {answer.text}")
            if answer.confidence < 1.0:
                print(f"(Confidence: {answer.confidence:.0%})")
            print()

        except KeyboardInterrupt:
            print("\n\n👋 Goodbye!")
            break
        except Exception as e:
            print(f"\n❌ Error: {e}\n")

    return orchestrator

# ============================================================================
# GUIDE D'UTILISATION COMPLET
# ============================================================================

"""
================================================================================
GRANDNEXUS PHASE 2 - GUIDE D'UTILISATION COMPLET
================================================================================

🎯 OBJECTIF:
Transformer GrandNexus en une IA intelligente capable de:
- Raisonner logiquement (déduction, induction)
- Apprendre de nouvelles connaissances
- Répondre à des questions
- S'auto-évaluer (métacognition)
- Planifier et exécuter des buts

📦 INSTALLATION (Google Colab):

1. Copier TOUS les codes Phase 2 (Part 1, 2, 3, 4) dans une cellule

2. Installer les dépendances optionnelles:
   !pip install wikipedia-api feedparser openai anthropic

3. Pour utiliser un LLM, définir la clé API:
   import os
   os.environ['OPENAI_API_KEY'] = 'your-key-here'

🚀 UTILISATION RAPIDE:

# Option 1: Test rapide (6 minutes)
orchestrator = await quick_demo()

# Option 2: Test complet (30 min)
orchestrator = await run_intelligent_grandnexus(duration_hours=0.5)

# Option 3: Avec LLM (nécessite clé API)
orchestrator = await run_with_llm(
    provider="openai",
    api_key="your-api-key"
)

# Option 4: Session interactive
orchestrator = await interactive_session()

🧪 TESTS DISPONIBLES:

# Test de raisonnement logique
orchestrator = await test_reasoning_capabilities()

# Test du système de buts
orchestrator = await test_goal_system()

# Test de métacognition
orchestrator = await test_metacognition()

# Test de conversation
orchestrator = await test_full_conversation()

# Benchmark complet
orchestrator = await benchmark_system()

📊 ANALYSE POST-EXÉCUTION:

# Statistiques générales
print("Phase 1:", orchestrator.graph.get_stats())
print("Phase 2:", orchestrator.reasoning_engine.get_stats())

# Auto-évaluation
assessment = await orchestrator.metacognitive_monitor.perform_self_assessment()
print(f"Self-awareness: {orchestrator.metacognitive_monitor.self_awareness_level:.2%}")

# Export de la base de connaissances
analyze_database()
export_knowledge_graph()

💡 EXEMPLES D'INTERACTION:

# Enseigner des faits
await orchestrator.query_interface.process_query("Socrates is a Human")
await orchestrator.query_interface.process_query("Humans are mortal")

# Poser des questions
answer = await orchestrator.query_interface.process_query("Is Socrates mortal?")
print(answer.text)  # → "Oui, Socrates est mortel (par inférence)"

# Créer un but
goal = await orchestrator.goal_agent.create_goal(
    description="Learn about quantum physics",
    priority=8
)
await orchestrator.goal_agent.execute_goal(goal.id)

# Vérifier si le système peut répondre
can_answer, confidence, explanation = await orchestrator.metacognitive_monitor.can_answer_question(
    "What is quantum mechanics?"
)

🎓 CONCEPTS AVANCÉS:

1. RAISONNEMENT SYMBOLIQUE:
   - Le système utilise des règles d'inférence (transitivité, héritage)
   - Peut déduire de nouveaux faits à partir de faits connus
   - Exemple: Si A→B et B→C, alors A→C

2. MÉTACOGNITION:
   - Le système évalue ses propres connaissances
   - Identifie ses lacunes
   - Suggère des améliorations
   - Niveau de "self-awareness" calculé

3. SYSTÈME DE BUTS:
   - Planification automatique d'actions
   - Exécution asynchrone
   - Gestion des priorités

4. INTERFACE LLM (optionnel):
   - Extraction sémantique avancée
   - Compréhension du langage naturel
   - Génération de réponses élaborées

📈 MÉTRIQUES CLÉS:

- knowledge_completeness: Complétude de la base de connaissances
- reasoning_capability: Capacité de raisonnement
- learning_efficiency: Efficacité d'apprentissage
- self_awareness_level: Niveau de conscience de soi
- cognitive_load: Charge cognitive actuelle

🔧 CONFIGURATION AVANCÉE:

config = RealSystemConfig()
config.CYCLE_INTERVAL_SECONDS = 1.0  # Vitesse des cycles
config.MAX_GRAPH_NODES = 10000       # Limite du graphe
config.MAX_HOT_CACHE = 1000          # Limite mémoire
config.LLM_PROVIDER = "openai"       # Provider LLM
config.LLM_API_KEY = "your-key"      # Clé API

orchestrator = IntelligentGrandNexusOrchestrator(config)
await orchestrator.initialize()
await orchestrator.run(duration_hours=2.0)

🎯 TESTS DE QI POSSIBLES MAINTENANT:

1. ✅ Analogies simples (via raisonnement)
2. ✅ Déduction logique (syllogismes)
3. ✅ Compréhension de relations (is_a, part_of)
4. ✅ Auto-évaluation des connaissances
5. ✅ Conscience des limitations

❌ PAS ENCORE POSSIBLES:
- Créativité
- Compréhension profonde du langage (sans LLM)
- Raisonnement causal complexe
- Apprentissage par analogie sophistiquée

📚 POUR ALLER PLUS LOIN:

1. Ajouter plus de règles d'inférence dans SymbolicReasoningEngine
2. Intégrer un LLM pour extraction sémantique avancée
3. Implémenter un système de croyances avec incertitude (Bayésien)
4. Ajouter un module de dialogue multi-tours
5. Créer un système d'explication (pourquoi le système a répondu X)

================================================================================
FIN DU GUIDE - GRANDNEXUS PHASE 2
================================================================================

🎉 FÉLICITATIONS!

Vous avez maintenant un système d'IA qui:
✅ Apprend de données réelles
✅ Raisonne logiquement
✅ S'auto-évalue
✅ Planifie et exécute des buts
✅ Dialogue avec l'utilisateur
✅ Possède une forme de "conscience de soi"

C'est un véritable "organisme cognitif" qui va bien au-delà d'un simple script!

Pour lancer immédiatement:
>>> orchestrator = await quick_demo()

ou pour une session interactive:
>>> orchestrator = await interactive_session()

Bonne exploration! 🚀
"""

'\n================================================================================\nGRANDNEXUS PHASE 2 - GUIDE D\'UTILISATION COMPLET\n================================================================================\n\n🎯 OBJECTIF:\nTransformer GrandNexus en une IA intelligente capable de:\n- Raisonner logiquement (déduction, induction)\n- Apprendre de nouvelles connaissances\n- Répondre à des questions\n- S\'auto-évaluer (métacognition)\n- Planifier et exécuter des buts\n\n📦 INSTALLATION (Google Colab):\n\n1. Copier TOUS les codes Phase 2 (Part 1, 2, 3, 4) dans une cellule\n\n2. Installer les dépendances optionnelles:\n   !pip install wikipedia-api feedparser openai anthropic\n\n3. Pour utiliser un LLM, définir la clé API:\n   import os\n   os.environ[\'OPENAI_API_KEY\'] = \'your-key-here\'\n\n🚀 UTILISATION RAPIDE:\n\n# Option 1: Test rapide (6 minutes)\norchestrator = await quick_demo()\n\n# Option 2: Test complet (30 min)\norchestrator = await run_intelligent_grandnexus(duration_hou

In [ ]:
# 1. Copier les 4 parties Phase 2 + les corrections Phase 1 dans une cellule

# 2. Lancer une démo rapide (6 minutes)
orchestrator = await quick_demo()

# 3. OU Session interactive
orchestrator = await interactive_session()
# Puis dialoguer:
# You: Socrates is a Human
# You: Humans are mortal
# You: Is Socrates mortal?
# AI: Oui, Socrates est mortel (par inférence)

# 4. OU Tests complets
orchestrator = await test_reasoning_capabilities()
orchestrator = await test_metacognition()
orchestrator = await benchmark_system()
```

## 🎯 **CAPACITÉS DÉBLOQUÉES**

**Raisonnement symbolique** - Peut déduire que "Socrate est mortel"
**Métacognition** - Sait ce qu'il sait et ce qu'il ne sait pas
**Système de buts** - Planifie et exécute des objectifs
**Dialogue intelligent** - Répond aux questions avec raisonnement
**Auto-évaluation** - Calcule son niveau de "conscience"

## 💬 **CONVERSATION POSSIBLE**
```
You: A dog is an Animal
AI: Thank you, I learned that dog is an Animal.

You: Animals are living beings
AI: Thank you, I learned that animals are living beings.

You: Is a dog a living being?
AI: Oui, dog est un living being (par inférence).
Confidence: 85%

In [ ]:
# Test logique
orchestrator = await test_reasoning_capabilities()
# → Score de raisonnement

# Test métacognitif
assessment = await orchestrator.metacognitive_monitor.perform_self_assessment()
# → Self-awareness level: 0.65 (65%)

# Benchmark complet
orchestrator = await benchmark_system()
# → TOTAL SCORE: 72.3/100 ✅ GOOD

In [ ]:
orchestrator = await quick_demo()

In [ ]:
# Test logique
orchestrator = await test_reasoning_capabilities()
# → Score de raisonnement

# Test métacognitif
assessment = await orchestrator.metacognitive_monitor.perform_self_assessment()
# → Self-awareness level: 0.65 (65%)

# Benchmark complet
orchestrator = await benchmark_system()
# → TOTAL SCORE: 72.3/100 ✅ GOOD